In [1]:
import pandas as pd
import numpy as np
from sklearn.utils import Bunch
from catboost import CatBoostClassifier,  Pool
# from geopy.distance import geodesic
import pickle

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
# warnings.filterwarnings("ignore", category=pd.core.generic.SettingWithCopyWarning)
warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)

from google.cloud import bigquery
from google.cloud import storage
import os
from datetime import datetime

path = r"C:\Users\Dwaipayan C\AppData\Roaming\gcloud\application_default_credentials.json"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

import sys
sys.path.append(r"D:\Officedata\Feature_Universe_Training")
from Utils.bq_fast_download import bq_to_df

DefaultCredentialsError: File C:\Users\Dwaipayan C\AppData\Roaming\gcloud\application_default_credentials.json was not found.

# Config

In [ ]:
# B Score model configuration
# ─────────────────────────────────────────────────────────────────────────────
from google.cloud import bigquery
import time
import pandas as pd
from datetime import datetime

# client = bigquery.Client()
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Schema and table name definitions (update OBDATE for each model run)
schema1 = 'risk_mart'
schema2 = 'worktable_data_analysis'
schema3 = 'dap_ds_poweruser_playground'

UTIL_PATH = f"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\DC_Model_Data_Sets_Feature_Universe\Feature_Universe_Training_Config"

# UTIL_PATH = F"D:\Officedata\Feature_Universe_Training"

GS_BUCKET = "prod-asia-southeast1-tonik-report-analytics"
GS_BUCKET_OLD = "prod-asia-southeast1-tonik-aiml-workspace"
PROJECT_ID = "prj-prod-dataplatform"

tab = 't_score_snapshot'
ver = 'v1'

OBDATE = '2026-10-27'   # Observation date — update for each scoring run
SDATE  = '2023-01-01'   # Start date for training window
EDATE  = OBDATE

formatted_date = datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')
ob_formatted   = datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')

al       = f'applied_loans_20230101_{ob_formatted}'
altrans  = f'applied_loans_20210701_{ob_formatted}_trans'
nal      = f'tsa_onboarded_but_never_applied_loan_20230101_{ob_formatted}'

# Intermediate staging tables
bscorebasedata = f'{tab}_base_data_{ver}_{datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')}_{datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')}'
print(bscorebasedata)
bscoresnapshotcustomerdata        = f'{tab}_applied_loans_customer_information_data_{ver}'
bscoresnapshotcustomertransactiondata = f'{tab}_applied_loans_customer_transaction_data_{ver}'
bscoresnapshotcustomerevent       = f'{tab}_applied_loans_customer_event_data_{ver}'
bscoresnapshotcontactability      = f'{tab}_applied_loans_contactability_{ver}'
bscoresnapshotcombineddata        = f'{tab}_model_applied_loans_{ver}_{ob_formatted}'


step61 = f'{schema2}.{tab}_vb_results_{ver}_{ob_formatted}'
step62 = f'{schema2}.{tab}_chatbot_{ver}_{ob_formatted}'
step63 = f'{schema2}.{tab}_chatbot_fb_{ver}_{ob_formatted}'
step64 = f'{schema2}.{tab}_chatbot_users_{ver}_{ob_formatted}'
step65 = f'{schema2}.{tab}_genesys_chat_{ver}_{ob_formatted}'
step66 = f'{schema2}.{tab}_genesys_email_{ver}_{ob_formatted}'
step67 = f'{schema2}.{tab}_inbound_voice_{ver}_{ob_formatted}'
step68 = f'{schema2}.{tab}_airudder_softcollection_calls_{ver}_{ob_formatted}'
step682 = f'{schema2}.{tab}_cs_vb_cnt_calls_total_{ver}_{ob_formatted}'
step69 = f'{schema2}.{tab}_reach_enhanced_{ver}_{ob_formatted}'
step610 = f'{schema2}.{tab}_ars_results_{ver}_{ob_formatted}'
step611 = f'{schema2}.{tab}_inbound_additional_features_{ver}_{ob_formatted}'
step612 = f'{schema2}.{tab}_outbound_voice_features_{ver}_{ob_formatted}'
step6 = f'{schema2}.{tab}_contact_features_extended_{ver}_{ob_formatted}'
step7 = f'{schema2}.{tab}_contact_genesysdata_{ver}_{ob_formatted}'
step8 = f'{schema2}.{tab}_loan_level_swathi_table_{ver}_{ob_formatted}'
step9 = f'{schema2}.{tab}_ptp_details_{ver}_{ob_formatted}'

step101 = f'{schema2}.{tab}_account_product_portfolio_composition_{ver}_{ob_formatted}'
step102 = f'{schema2}.{tab}_activity_recency_dormancy_{ver}_{ob_formatted}'
step103 = f'{schema2}.{tab}_tx_balance_liquidity_{ver}_{ob_formatted}'
step104 = f'{schema2}.{tab}_bureau_external_scoring_{ver}_{ob_formatted}'
step105 = f'{schema2}.{tab}_completion_settlement_metrics_{ver}_{ob_formatted}'
step106 = f'{schema2}.{tab}_conversion_campaign_engagement_events_{ver}_{ob_formatted}'
step107 = f'{schema2}.{tab}_customer_acquisition_retention_{ver}_{ob_formatted}'
step108 = f'{schema2}.{tab}_customer_lifecycle_engagement_{ver}_{ob_formatted}'
step109 = f'{schema2}.{tab}_demographic_verification_enhanced_{ver}_{ob_formatted}'
step1010 = f'{schema2}.{tab}_deposit_savings_products_{ver}_{ob_formatted}'
step1011 = f'{schema2}.{tab}_derived_ratio_metrics_{ver}_{ob_formatted}'
step1012 = f'{schema2}.{tab}_fraud_compliance_indicators_{ver}_{ob_formatted}'
step1013 = f'{schema2}.{tab}_loan_product_specific_metrics_{ver}_{ob_formatted}'
step1014 = f'{schema2}.{tab}_loan_staging_milestones_{ver}_{ob_formatted}'
step10151 = f'{schema2}.{tab}_payment_behaviour_advanced_{ver}_{ob_formatted}'
step10152 = f'{schema2}.{tab}_payment_behaviour_advanced_2_{ver}_{ob_formatted}'
step1016 = f'{schema2}.{tab}_revenue_fee_decomposition_{ver}_{ob_formatted}'
step1017 = f'{schema2}.{tab}_risk_segmentation_categorization_{ver}_{ob_formatted}'
step1018 = f'{schema2}.{tab}_time_window_cohort_features_{ver}_{ob_formatted}'
step1019 = f'{schema2}.{tab}_transaction_volume_type_breakdown_{ver}_{ob_formatted}'
step111 = f'{schema2}.{tab}_collection_call_features_{ver}_{ob_formatted}'
step112 = f'{schema2}.{tab}_payhinga_features_{ver}_{ob_formatted}'

step12c = f'{schema2}.PH_City_Province_Event_Mapping_{ver}_{ob_formatted}'

step12d = f'{schema2}.{tab}_ph_shock_feature_{ver}_{ob_formatted}'
step14 = f'{schema2}.{tab}_prevloan_less_6dpd_details_{ver}_{ob_formatted}'

step17 = f'{schema2}.{tab}_customer_level_5dpd_details_{ver}_{ob_formatted}'

step20 = f'dap_ds_poweruser_playground.{tab}_yuvashri_features_{ver}_{ob_formatted}'

step22 = f'{schema2}.{tab}_finscore_{ver}_{ob_formatted}'
step23 = f'{schema2}.{tab}_trustingsocialscore_{ver}_{ob_formatted}'
step24 = f'{schema2}.{tab}_bureauinc_score_{ver}_{ob_formatted}'

for step_name, step_value in {
    "step61": step61,
    "step62": step62,
    "step63": step63,
    "step64": step64,
    "step65": step65,
    "step66": step66,
    "step67": step67,
    "step68": step68,
    "step682": step682,
    "step69": step69,
    "step610": step610,
    "step611": step611,
    "step612": step612,
    "step6": step6,
    "step7": step7,
    "step8": step8,
    "step9": step9,
    "step101": step101,
    "step102": step102,
    "step103": step103,
    "step104": step104,
    "step105": step105,
    "step106": step106,
    "step107": step107,
    "step108": step108,
    "step109": step109,
    "step1010": step1010,
    "step1011": step1011,
    "step1012": step1012,
    "step1013": step1013,
    "step1014": step1014,
    "step10151": step10151,
    "step10152": step10152,
    "step1016": step1016,
    "step1017": step1017,
    "step1018": step1018,
    "step1019": step1019,
    "step111": step111,
    "step112": step112,
    "step12c": step12c,
    "step12d": step12d,
    "step14": step14,
    "step17": step17,
    "step20": step20,
    "step22": step22,
    "step23": step23,
    "step24": step24,
}.items():
    print(f"{step_name}: {step_value}")



# Final output tables (split by loan type)
trench3_b_score_model_applied_loans = f'{tab}_model_applied_loans_app_package_{ver}_{ob_formatted}'

print(f"OBDATE: {OBDATE}")
print(f"Output table: {schema2}.{bscoresnapshotcombineddata}")

print(f"Applied Table: {schema1}.{al}")
print(f"Applied Trans Table: {schema1}.{altrans}")
print(f"tsa_onboarded_but_never_applied_loan Table: {schema1}.{nal}")

APP_PACKAGE = f'{tab}_applied_all_dis_loan_app_package_{ver}_{ob_formatted}'
print(f"App Package Table: {schema2}.{APP_PACKAGE}")

App_Feature_Table = f'{tab}_model_app_features_{ver}_{ob_formatted}'
print(f"App Feature Table: {schema2}.{App_Feature_Table}")

cic_feature_table = f'{tab}_cic_features_{ver}_{ob_formatted}'
print(f"CIC Feature Table: {schema2}.{tab}_cic_features_{ver}_{ob_formatted}")

android_devise_score_features = f"{tab}_android_device_score_feature_list_{ver}_{ob_formatted}"
print(f"Android Device Score Feature Table: {schema2}.{android_devise_score_features}")

ios_device_score_features = f"{tab}_ios_device_score_features_{ver}_{ob_formatted}"
print(f"iOS Device Score Feature Table: {schema2}.{ios_device_score_features}")

nitin_event_features = f"{schema2}.{tab}_events_feats_{ver}_{ob_formatted}"
print(f"nitin_event_features Table: {nitin_event_features}")

demo_features = f"{schema3}.{tab}_demo_features_{ver}"
print(f"Demo Feature Table: {demo_features}")
stepfinaltab = f"{schema2}.{tab}_model_full_features_v7_{ver}_{ob_formatted}"
print(f"Final Feature Table: {stepfinaltab}")

finaltab = f"{schema2}.{tab}_model_full_features_model_dataset_{ver}_{ob_formatted}"
print(f"Final Table: {finaltab}")

In [ ]:
sq = f"""
create or replace table worktable_data_analysis.applied_all_dis_credo_device_info as
(WITH base_data AS (
  SELECT *
  FROM {schema2}.{bscorebasedata}
  WHERE DATE(ln_loan_appln_time) BETWEEN '2024-09-01' AND '{OBDATE}'
),
credolab_track AS (
  SELECT
    CAST(B.cust_id AS INT64) AS customerid,
    A.refno,
    datetime(A.createdOn) createdOn
  FROM `dl_loans_db_raw.tdbk_credolab_track` A
  JOIN `dl_customers_db_raw.tdbk_customer_mtb` B
    ON A.userId = B.user_id
),
credolab_android AS (
  SELECT
    A.customerid,
    B.referenceNumber AS refno,
    B.requestedDate AS createdOn
  FROM `risk_credit_mis.loan_master_table` A
  JOIN (
    SELECT
      digitalLoanAccountId,
      referenceNumber,
      requestedDate
    FROM `credolab_raw.credolab_insights_android`
    WHERE referenceNumber NOT IN (
      SELECT refno FROM `dl_loans_db_raw.tdbk_credolab_track`
    )
  ) B
  ON A.digitalLoanAccountId = B.digitalLoanAccountId
),
insight_dl AS (
  SELECT * FROM credolab_track
  UNION ALL
  SELECT * FROM credolab_android
)
SELECT DISTINCT
  A.customerid,
  loan.digitalLoanAccountId,
  insight_dl.refno,
  TIMESTAMP(insight_dl.createdOn, 'Asia/Manila') AS credo_inquiry_date_Manila,
  A.ln_loan_appln_time,Observation_date
FROM base_data A
JOIN `risk_credit_mis.loan_master_table` loan
  ON A.digitalLoanAccountId = loan.digitalLoanAccountId
LEFT JOIN insight_dl
  ON loan.customerId = insight_dl.customerid
WHERE DATE(insight_dl.createdOn) <= A.Observation_date
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY loan.digitalLoanAccountId
  ORDER BY insight_dl.createdOn DESC
) = 1)
;
"""
job = client.query(sq)
job.result()  # Wait for the job to complete.
print(f'Table {schema2}.{bscorebasedata} created successfully')
print("Total columns:", client.get_table(f"{schema2}.{bscorebasedata}").schema.__len__())

In [ ]:
ios_query = f"""
   
SELECT
  ios.*,
  dev.customerid,dev.Observation_date
FROM `credolab_raw.ios_credolab_datasets_struct_columns` AS ios

INNER JOIN `worktable_data_analysis.applied_all_dis_credo_device_info` AS dev
ON ios.deviceId = dev.refno
 
  """

# Run the query and save results to a Pandas DataFrame
# df_ios = client.query(ios_query).to_dataframe(progress_bar_type='tqdm')

df_ios = bq_to_df(ios_query)
print(f"The shape of the dataframe is: {df_ios.shape}")

# df_ios.to_csv('apps_score.csv')

In [ ]:
# df_ios = df_ios[df_ios['l'] ==1]

###  <span style='background :yellow' > Flatten the data </span>

In [ ]:
dict_cols = ['GeneralInfo', 'GeneralData','GeoInfo']

for col in dict_cols:
    expanded = pd.json_normalize(df_ios[col]).add_prefix(f'{col}_')
    df_ios = df_ios.drop(columns=[col]).join(expanded)

In [ ]:
df_device = pd.read_csv(r'D:\Officedata\Feature_Universe_Training\Support_Data\mobile_device_with_gsm.csv')

In [ ]:
df_device[['device_model','device_name']]

## Device mapping 

In [ ]:
# normalize both sides
df_ios['model_l'] = df_ios['GeneralInfo_model'].str.lower().str.strip()
df_device['model_l'] = df_device['device_model'].str.lower().str.strip()

# create mapping dict (device_model → device_name)
model_to_name = (
    df_device
    .drop_duplicates('model_l')   # safety
    .set_index('model_l')['device_name']
)

# map
df_ios['device_name'] = df_ios['model_l'].map(model_to_name)


In [ ]:
df_ios['device_name']

In [ ]:
# df_ios['GeneralInfo_model'].value_counts()

In [ ]:
df_ios['device_name_l'] = df_ios['device_name'].str.lower().str.strip()


In [ ]:
df_ios['device_tier'] = np.select(
    [
        df_ios['device_name_l'].str.contains('pro max', na=False),
        df_ios['device_name_l'].str.contains('pro', na=False),
        df_ios['device_name_l'].str.contains('plus', na=False),
        df_ios['device_name_l'].str.contains('se|mini', na=False),
        df_ios['device_name_l'].str.contains('ipad', na=False)
    ],
    [
        'premium',
        'premium',
        'upper_mid',
        'budget',
        'tablet'
    ],
    default='mid'
)


In [ ]:
df_ios['device_tier'].value_counts(normalize=True)

In [ ]:
import re
def extract_generation(name):
    if pd.isna(name):
        return np.nan
    
    # iPhone 11 / 12 / 13 / 14 / 15
    m = re.search(r'iphone\s*(\d+)', name)
    if m:
        return int(m.group(1))
    
    # SE (2020 / 2022)
    m = re.search(r'se\s*\((\d{4})\)', name)
    if m:
        return int(m.group(1))
    
    return np.nan

df_ios['device_generation'] = df_ios['device_name_l'].apply(extract_generation)


In [ ]:
df_ios['device_generation'].value_counts()

In [ ]:
df_ios['device_age_bucket'] = np.select(
    [
        df_ios['device_generation'] >= 15,
        df_ios['device_generation'].between(13, 14),
        df_ios['device_generation'].between(11, 12),
        df_ios['device_generation'] <= 10
    ],
    [
        'new',
        'recent',
        'old',
        'very_old'
    ],
    default='unknown'
)


In [ ]:
df_ios['device_age_bucket'].value_counts()

In [ ]:
df_ios['is_pro_model'] = df_ios['device_name_l'].str.contains('pro', na=False).astype(int)
df_ios['is_max_model'] = df_ios['device_name_l'].str.contains('max', na=False).astype(int)
df_ios['is_large_variant'] = df_ios['device_name_l'].str.contains('plus|max', na=False).astype(int)


## Country code


In [ ]:
# GeneralData_currency_code
# = the currency configured on the iOS device / Apple ecosystem


In [ ]:
asia = ['PHP','JPY','CNY','KRW','SGD','HKD','TWD','MYR','THB','VND','IDR','INR','KHR']
middle_east = ['AED','SAR','QAR','KWD','BHD','OMR','JOD']
europe = ['EUR','GBP','CHF','SEK','NOK','DKK','PLN','CZK','HUF','RON','ISK']
americas = ['USD','CAD','BRL','ARS','CRC','BSD','JMD','KYD','GTQ']
africa = ['ZAR','GHS','XAF','XOF','AOA','DZD','MAD']
oceania = ['AUD','NZD']

def currency_region(c):
    if c in asia: return 'asia'
    if c in middle_east: return 'middle_east'
    if c in europe: return 'europe'
    if c in americas: return 'americas'
    if c in africa: return 'africa'
    if c in oceania: return 'oceania'
    return 'other'
df_ios['GeneralData_currency_code'] = df_ios['GeneralData_currency_code'].str.upper().str.strip()

df_ios['currency_region'] = df_ios['GeneralData_currency_code'].apply(currency_region)


In [ ]:
df_ios['currency_region'].value_counts()

In [ ]:
high_income = [
    'USD','EUR','GBP','CHF','AUD','NZD','SGD','HKD',
    'CAD','JPY','KRW','NOK','SEK','DKK'
]

df_ios['is_high_income_currency'] = df_ios['GeneralData_currency_code'].isin(high_income).astype(int)


In [ ]:
df_ios['is_high_income_currency'].value_counts()

In [ ]:
df_ios['GeneralData_locale_identifier'].value_counts()

In [ ]:
df_ios[['customerid','GeneralInfo_model','Observation_date',
'device_name',
'device_tier',
'device_generation',
'device_age_bucket',
'is_pro_model',
'is_max_model',
'is_large_variant',
'GeneralData_currency_code',
'currency_region',
'is_high_income_currency',
'GeneralData_locale_identifier',
'GeneralData_collator_identifier',
'GeneralData_total_memory',
'GeneralData_free_memory',
'GeneralData_physical_memory',]].info()

In [ ]:
def rename_col(col):
    if col in ['customerid', 'Observation_date']:
        return col
    elif col.startswith('device'):
        return col
    else:
        return f"device_{col}"

df_ios = df_ios.rename(columns={col: rename_col(col) for col in df_ios.columns})

In [ ]:
df_ios.columns

In [ ]:
cols = [
    'customerid',
    'device_GeneralInfo_model',
    'Observation_date',
    'device_name',
    'device_tier',
    'device_generation',
    'device_age_bucket',
    'device_is_pro_model',
    'device_is_max_model',
    'device_is_large_variant',
    'device_GeneralData_currency_code',
    'device_currency_region',
    'device_is_high_income_currency',
    'device_GeneralData_locale_identifier',
    'device_GeneralData_collator_identifier',
    'device_GeneralData_total_memory',
    'device_GeneralData_free_memory',
    'device_GeneralData_physical_memory',
]

In [ ]:
df_ios[cols].info()

In [ ]:
from google.cloud import bigquery



schema = [
    bigquery.SchemaField("customerid", "INT64"),
    bigquery.SchemaField("device_GeneralInfo_model", "STRING"),
    bigquery.SchemaField("Observation_date", "DATE"),
    bigquery.SchemaField("device_name", "STRING"),
    bigquery.SchemaField("device_tier", "STRING"),
    bigquery.SchemaField("device_generation", "FLOAT64"),
    bigquery.SchemaField("device_age_bucket", "STRING"),
    bigquery.SchemaField("device_is_pro_model", "INT64"),
    bigquery.SchemaField("device_is_max_model", "INT64"),
    bigquery.SchemaField("device_is_large_variant", "INT64"),
    bigquery.SchemaField("device_GeneralData_currency_code", "STRING"),
    bigquery.SchemaField("device_currency_region", "STRING"),
    bigquery.SchemaField("device_is_high_income_currency", "INT64"),
    bigquery.SchemaField("device_GeneralData_locale_identifier", "STRING"),
    bigquery.SchemaField("device_GeneralData_collator_identifier", "STRING"),
    bigquery.SchemaField("device_GeneralData_total_memory", "FLOAT64"),
    bigquery.SchemaField("device_GeneralData_free_memory", "FLOAT64"),
    bigquery.SchemaField("device_GeneralData_physical_memory", "FLOAT64"),
]

from google.cloud import bigquery

client = bigquery.Client(project="prj-prod-dataplatform")

dataset_id = "risk_mart"
table_id = f"{ios_device_score_features}"

table_ref = client.dataset(dataset_id).table(table_id)

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(
    df_ios[cols],   # ensure only required columns are loaded
    table_ref,
    job_config=job_config
)

job.result()
print(f"Table {dataset_id}.{table_id} created successfully")

